In [5]:
import cv2
import numpy as np
import pandas as pd
import os

def extract_features(image_path):
    """
    Extract features from an image.
    Features include: color histogram.
    """
    # Read the image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Unable to read image {image_path}")
        return None

    # Resize the image to a fixed size
    image = cv2.resize(image, (128, 128))
    
    # Convert to HSV color space
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Compute color histogram (32 bins for H, S, V channels)
    h_hist = cv2.calcHist([hsv_image], [0], None, [32], [0, 256])
    s_hist = cv2.calcHist([hsv_image], [1], None, [32], [0, 256])
    v_hist = cv2.calcHist([hsv_image], [2], None, [32], [0, 256])

    # Normalize histograms
    h_hist = cv2.normalize(h_hist, h_hist).flatten()
    s_hist = cv2.normalize(s_hist, s_hist).flatten()
    v_hist = cv2.normalize(v_hist, v_hist).flatten()

    # Combine all histograms into a single feature vector
    feature_vector = np.hstack([h_hist, s_hist, v_hist])
    return feature_vector

def process_images_in_directory(directory_path, output_csv):
    """
    Process all images in a directory and save extracted features to a CSV.
    """
    # Prepare a list for storing features
    features_list = []
    file_names = []

    # Iterate over images in the directory
    for file_name in os.listdir(directory_path):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, file_name)
            features = extract_features(image_path)
            if features is not None:
                features_list.append(features)
                file_names.append(file_name)

    # Handle the case where no valid features were extracted
    if not features_list:
        print("No features were extracted. Please check the input directory or the feature extraction process.")
        return

    # Convert the feature list to a DataFrame
    feature_columns = [f'feature_{i}' for i in range(len(features_list[0]))]
    df = pd.DataFrame(features_list, columns=feature_columns)
    df.insert(0, 'image_name', file_names)  # Add image names as the first column

    # Save DataFrame to CSV
    df.to_csv(output_csv, index=False)
    print(f"Features saved to {output_csv}")

# Example usage
directory_path = 'D:\\7th Semester\\ML\\ML Project\\Crop Images 1\\PlantVillage\\Pepper__bell___healthy'
output_csv = 'image_features 1.csv'

process_images_in_directory(directory_path, output_csv)


Features saved to image_features 1.csv
